# 🤖 Machine Learning - Agrupamento e Segmentação de Perfis com K-Means

**Autor:** Victor Oliveira  
**Pós-Graduação:** Ciência de Dados e Inteligência Artificial (Universidade São Judas)  
**Stack:** Python, `Scikit-Learn`, `Pandas`, `NumPy`, `Matplotlib`, `Seaborn`  

---

## 🎯 Objetivos do Estudo
1. **Entender a Aprendizagem Não-Supervisionada**: Aplicação do algoritmo **K-Means** para identificar agrupamentos naturais de dados.
2. **Determinar o Número Ideal de Clusters (K)**: Aplicação da métrica de **Inércia** e o **Método do Cotovelo (*Elbow Method*)**.
3. **Localização Exata dos Centróides**: Mapear as coordenadas matemáticas centrais de cada grupo.
4. **Avaliação da Qualidade do Agrupamento**: Cálculo e interpretação do **Score de Silhueta (*Silhouette Score*)**.

### 1. Importação de Bibliotecas e Geração dos Dados

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Geração de dados de exemplo representando agrupamentos de comportamento
blob_centers = np.array([
    [0.2, 2.3],
    [-1.5, 2.3],
    [-2.8, 1.8],
    [-2.8, 2.8],
    [-2.8, 0.8]
])
blob_std = np.array([0.4, 0.3, 0.1, 0.1, 0.1])
X, y_true = make_blobs(n_samples=2000, centers=blob_centers, cluster_std=blob_std, random_state=42)

print(f"Dataset gerado com {X.shape[0]} amostras e {X.shape[1]} dimensões.")

### 2. Escolha do K Ideal: O Método do Cotovelo (*Elbow Method*)

In [2]:
inercias = []
k_range = range(1, 10)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X)
    inercias.append(kmeans.inertia_)

# Plot do Método do Cotovelo
plt.figure(figsize=(9, 5))
plt.plot(k_range, inercias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Número de Clusters (k)', fontsize=12)
plt.ylabel('Inércia (Soma das Distâncias Quadráticas)', fontsize=12)
plt.title('Método do Cotovelo (Elbow Method)', fontsize=13, fontweight='bold')
plt.axvline(x=5, color='r', linestyle='--', label='K Ideal = 5')
plt.legend()
plt.show()

### 3. Mapeamento das Coordenadas dos Centróides e Fronteiras de Decisão

In [3]:
kmeans_final = KMeans(n_clusters=5, random_state=42, n_init=10)
labels = kmeans_final.fit_predict(X)
centroides = kmeans_final.cluster_centers_

# Criando a malha espacial para desenhar as fronteiras de decisão dos centróides
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))
Z = kmeans_final.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(11, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='tab10')
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='tab10', s=10, alpha=0.5)
plt.scatter(centroides[:, 0], centroides[:, 1], c='red', marker='X', s=250, edgecolor='black', linewidth=1.5, label='Centróides')

# Anotando as coordenadas de cada centróide
for i, (cx, cy) in enumerate(centroides):
    plt.annotate(f'Centróide {i+1}\n({cx:.2f}, {cy:.2f})', xy=(cx, cy), xytext=(cx + 0.15, cy + 0.15),
                 bbox=dict(boxstyle='round,pad=0.3', fc='yellow', alpha=0.8),
                 arrowprops=dict(arrowstyle='->', color='black'), fontsize=9, fontweight='bold')

plt.title('Fronteiras de Decisão & Localização Exata dos Centróides', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.show()

### 4. Avaliação pelo Score de Silhueta e Conclusões

In [4]:
sil_score = silhouette_score(X, labels)
print(f"Score de Silhueta (Silhouette Score): {sil_score:.4f}")